# Video Preprocess

In [16]:
import cv2
import json
import numpy as np
import os
from tqdm import tqdm
import mediapipe as mp

In [29]:
# VID_DIR     = "../../datasets/ISL_Gifs"
# INV_GLOSS   = "../../datasets/invGlossList.json"    # {"hi": ["vid1","vid7",...], ...}
VID_DIR = r'D:\Work\text2sign\datasets\WLASL\videos'
INV_GLOSS = '../../datasets/WLASL/WLASL_converted.json'
OUT_DIR     = "../../datasets/sl2t_data/data"
PRANJALSIR_DATA = "../../datasets/sl2t_data/data_pranjalSir/data"
SEQUENCE_LEN = 20
FEATURE_DIM  =  (33 + 21 + 21) * 3

### Make data dirs

In [30]:
import cv2
import numpy as np
import mediapipe as mp
import os
import json


mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=2,        
    smooth_landmarks=True,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6,
)

use_cuda = cv2.cuda.getCudaEnabledDeviceCount() > 0
if use_cuda:
    # Pre‑create GPU transform objects
    cuda_clahe = cv2.cuda.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cuda_bilateral = lambda src: cv2.cuda.bilateralFilter(src, d=5, sigmaColor=75, sigmaSpace=75)
    cuda_resizer = lambda src, sz: cv2.cuda.resize(src, sz, interpolation=cv2.INTER_CUBIC)

def preprocess_frame_gpu(img, target_size=(256,256)):
    # upload to GPU
    gpu = cv2.cuda_GpuMat()
    gpu.upload(img)

    # resize
    gpu = cuda_resizer(gpu, target_size)

    # convert BGR→LAB on GPU
    gpu_lab = cv2.cuda.cvtColor(gpu, cv2.COLOR_BGR2Lab)
    # split channels
    l_gpu, a_gpu, b_gpu = cv2.cuda.split(gpu_lab)

    # CLAHE on L channel
    l_eq_gpu = cuda_clahe.apply(l_gpu)
    lab_eq_gpu = cv2.cuda.merge([l_eq_gpu, a_gpu, b_gpu])

    # convert back LAB→BGR
    gpu_bgr = cv2.cuda.cvtColor(lab_eq_gpu, cv2.COLOR_Lab2BGR)

    # bilateral filter
    gpu_out = cuda_bilateral(gpu_bgr)

    # download to host
    return gpu_out.download()


#––– PREPROCESSING HELPERS –––#
def preprocess_frame(img, target_size=(256, 256)):
    """
    1. Upscale to a fixed resolution
    2. Apply CLAHE on the L-channel to boost local contrast
    3. Denoise with a bilateral filter
    """
    # resize (upsample or downsample)
    img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_CUBIC)

    # convert to LAB and apply CLAHE on L channel
    lab = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    img_eq = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

    # bilateral filter to smooth noise but keep edges
    img_denoised = cv2.bilateralFilter(img_eq, d=5, sigmaColor=75, sigmaSpace=75)
    return img_denoised

#––– KEYPOINT EXTRACTION –––#
def extract_kp(img):
    """Returns a FEATURE_DIM list of x,y,z landmarks (zero‑padded if missing)."""
    # preprocess to improve low‑res details
    # img_p = preprocess_frame(img)
    img_p = preprocess_frame_gpu(img) if use_cuda else preprocess_frame(img)

    img_rgb  = cv2.cvtColor(img_p, cv2.COLOR_BGR2RGB)
    res = holistic.process(img_rgb)
    kp = []

    # order: pose, left hand, right hand
    for lm_list, count in [
        (res.pose_landmarks, 33),
        (res.left_hand_landmarks, 21),
        (res.right_hand_landmarks,21)
    ]:
        if lm_list:
            for lm in lm_list.landmark:
                kp.extend([lm.x, lm.y, lm.z])
        else:
            kp.extend([0.0] * (count * 3))

    # pad/truncate to FEATURE_DIM
    # print(f"KP: {len(kp)}")
    if len(kp) < FEATURE_DIM:
        kp += [0.0] * (FEATURE_DIM - len(kp))
    else:
        kp = kp[:FEATURE_DIM]

    return kp

#––– LOAD AND PREPARE OUTPUT DIRS –––#
with open(INV_GLOSS, 'r') as f:
    gloss_map = json.load(f)

actions = sorted(gloss_map.keys())
label_map = {act: i for i, act in enumerate(actions)}
video2label = {vid: label_map[gloss]
               for gloss, vids in gloss_map.items() for vid in vids}

# ensure output dirs exist
for act in actions:
    os.makedirs(os.path.join(OUT_DIR, act), exist_ok=True)

for act in actions:
    os.makedirs(os.path.join(OUT_DIR, act), exist_ok=True)

#––– USAGE EXAMPLE –––#
# cap = cv2.VideoCapture("low_res_video.mp4")
# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret: break
#     keypoints = extract_kp(frame) # feed keypoints in model
# cap.release()


### Process Each videos

In [31]:
from datetime import datetime

log_dir = "logs"
log_file = os.path.join(log_dir, "file_check.log")
os.makedirs(log_dir, exist_ok=True)

for vid_id, label in tqdm(video2label.items(), total=len(video2label)):
    frames = []
    # print(f"vid id: {vid_id}")
    path = os.path.join(VID_DIR, vid_id + '.mp4')
    exists = os.path.isfile(path)
    ext = os.path.splitext(vid_id)[1].lower()

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] File '{path}' exists: {exists}\n"
    
    with open(log_file, "a") as f:
        f.write(log_message)

    if ext in ['.jpg', '.jpeg', '.png']:
        img = cv2.imread(path)
        if img is None:
            continue  
        frames.append(extract_kp(img))
    else:
        cap = cv2.VideoCapture(path)
        # print(f"Path: {path}")
        while True:
            ret, img = cap.read()
            # print(f"=======> ret: {ret}  .......... img: {img}")
            if not ret:
                break
            frames.append(extract_kp(img))
        cap.release()

    # pad/truncate sequence length
    # print(f"len frames: {len(frames)}")
    if len(frames) < SEQUENCE_LEN:
        frames += [[0.0]*FEATURE_DIM] * (SEQUENCE_LEN - len(frames))
    else:
        frames = frames[:SEQUENCE_LEN]

    # save to .npy
    out_dir = os.path.join(OUT_DIR, actions[label])
    idx = len(os.listdir(out_dir))
    np.save(os.path.join(out_dir, f"{idx}.npy"), np.array(frames))

100%|██████████| 21083/21083 [1:18:05<00:00,  4.50it/s]


### check npy

In [32]:
import os
import numpy as np

zero_count2 = 0
total2 = 0
for gloss in os.listdir(PRANJALSIR_DATA):
    gloss_path = os.path.join(PRANJALSIR_DATA, gloss)
    for fname in os.listdir(gloss_path):
        fname_path = os.path.join(gloss_path, fname)
        for file in os.listdir(fname_path):
            path = os.path.join(fname_path, file)
            arr = np.load(path)
            total2 += 1
            if np.all(arr == 0):
                zero_count2 += 1

print(f"Total files: {total2}")
print(f"Zero-only files: {zero_count2}")
print(f"Percent zeros: {100 * zero_count2 / total2:.2f}%")

Total files: 3600
Zero-only files: 2374
Percent zeros: 65.94%


In [33]:
import os
import numpy as np

zero_count = 0
total = 0
for gloss in tqdm(actions):
    gloss_dir = os.path.join(OUT_DIR, gloss)
    # print(" === ")
    # print(f"Gloss Dir: {gloss_dir}")
    for fname in os.listdir(gloss_dir):
        # print(f"File Name: {fname}")
        path = os.path.join(gloss_dir, fname)
        # print(f"Path: {path}")
        # print(" --- ")
        arr = np.load(path)
        total += 1
        if np.all(arr == 0):
            zero_count += 1

print(f"Total files: {total}")
print(f"Zero-only files: {zero_count}")
print(f"Percent zeros: {100 * zero_count / total:.2f}%")


100%|██████████| 2000/2000 [00:08<00:00, 237.11it/s]

Total files: 21239
Zero-only files: 21201
Percent zeros: 99.82%


# Model Arch and Training Loop

In [34]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Bidirectional, LSTM, Dropout, Dense, Layer
)
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

class Attention(Layer):
    def build(self, input_shape):
        # input_shape: (batch, time, features)
        self.W = self.add_weight(
            name='att_weight',
            shape=(input_shape[-1], 1),
            initializer='random_normal',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        # x: (batch, time, features)
        # e = tanh(x · W) => (batch, time, 1)
        e = tf.math.tanh(tf.tensordot(x, self.W, axes=[[2], [0]]))
        # alpha = softmax(e) over time axis => (batch, time, 1)
        alpha = tf.nn.softmax(e, axis=1)
        # context = sum over time of (x * alpha) => (batch, features)
        context = tf.reduce_sum(x * alpha, axis=1)
        return context

    def compute_output_shape(self, input_shape):
        # returns (batch, features)
        return (input_shape[0], input_shape[2])


# --- model configuration ---
SEQUENCE_LEN = 20
FEATURE_DIM  = 126
NUM_CLASSES  = len(actions) 

# --- build ---
model = Sequential([
    Input(shape=(SEQUENCE_LEN, FEATURE_DIM)),

    # Conv block 1
    Conv1D(64, 3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),

    # Conv block 2
    Conv1D(128, 3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),

    # Bidirectional LSTM stack
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),

    # Attention aggregation
    Attention(),
    Dropout(0.3),

    # Classification head
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

# compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint("best_model_CNN1s-LSTM.h5", save_best_only=True, monitor='val_loss'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
]

# history = model.fit(
#     X_train, Y_train,
#     validation_data=(X_test, Y_test),
#     epochs=100,
#     batch_size=32,
#     callbacks=callbacks
# )


In [35]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

# configs
SEQUENCE_LEN = 20
FEATURE_DIM  = 126
NUM_CLASSES  = len(actions)
OUT_DIR      = "../../datasets/sl2t_data/data"

def fix_array(arr):
    """
    Take an array of shape (SEQUENCE_LEN, D) and:
        - if D < FEATURE_DIM, pad zeros to the right
        - if D > FEATURE_DIM, truncate columns past FEATURE_DIM
    Returns an array of shape (SEQUENCE_LEN, FEATURE_DIM).
    """
    seq, dim = arr.shape
    if dim < FEATURE_DIM:
        # pad with zeros
        pad = np.zeros((seq, FEATURE_DIM - dim), dtype=arr.dtype)
        return np.concatenate([arr, pad], axis=1)
    elif dim > FEATURE_DIM:
        # truncate extra features
        return arr[:, :FEATURE_DIM]
    else:
        return arr

# gather & fix all samples
X, Y = [], []
for label, gloss in enumerate(actions):
    gloss_dir = os.path.join(OUT_DIR, gloss)
    for fname in os.listdir(gloss_dir):
        arr = np.load(os.path.join(gloss_dir, fname))  # shape (20, D)
        if arr.shape[0] != SEQUENCE_LEN:
            # if for some reason the time‑axis is off, skip
            continue
        arr_fixed = fix_array(arr)
        X.append(arr_fixed)
        Y.append(label)

# now X is a list of (20,126) arrays → safe to stack
X = np.stack(X, axis=0)                              # (N, 20, 126)
Y = to_categorical(Y, num_classes=NUM_CLASSES)

# stratified split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

# and train
history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)


Epoch 1/50
595/598 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0020 - loss: 7.5967

598/598 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.0020 - loss: 7.5966 - val_accuracy: 0.0033 - val_loss: 7.6000 - learning_rate: 0.0010
Epoch 2/50
598/598 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.0038 - loss: 7.5621 - val_accuracy: 0.0033 - val_loss: 7.6234 - learning_rate: 0.0010
Epoch 3/50
598/598 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.0039 - loss: 7.5446 - val_accuracy: 0.0033 - val_loss: 7.8692 - learning_rate: 0.0010
Epoch 4/50
598/598 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.0043 - loss: 7.5458 - val_accuracy: 0.0056 - val_loss: 7.6327 - learning_rate: 0.0010
Epoch 5/50
598/598 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.0036 - loss: 7.5373 - val_accuracy: 0.0052 - val_loss: 7.6195 - learning_rate: 0.0010
Epoch 6/50
598/598 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.0038 - loss: 7.5337 - val_accuracy: 0.0052 - val_loss: 7.6257 - learning_rate: 0.0010
Epoch 7/50
598/598 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.0036 - loss: 7.5334 - va

KeyboardInterrupt: 